# SECOM Semiconductor Process Data — Step 1

이 노트북은 **반도체 제조 공정 데이터의 구조를 이해하고 데이터 품질을 확인하는 첫 단계**입니다.

오늘 확인할 것:
1. 데이터가 정상적으로 로드되는가?
2. 샘플과 공정 변수는 몇 개인가?
3. Pass / Fail은 얼마나 불균형한가?
4. 결측치는 얼마나 많은가?
5. 값이 변하지 않는 상수 변수는 몇 개인가?

> SECOM의 feature는 익명화되어 있으므로 특정 feature를 온도·압력·유량 등으로 임의 해석하지 않습니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## 1. 데이터 불러오기

UCI 원본 URL을 먼저 사용하고, 연결 문제가 있을 때만 공개 GitHub mirror를 fallback으로 사용합니다.


In [ ]:
UCI_DATA = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom.data"
UCI_LABEL = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom_labels.data"

MIRROR_DATA = "https://raw.githubusercontent.com/halinh1512/Predictive-Model-Semiconductor-SECOM/master/Data/secom.data"
MIRROR_LABEL = "https://raw.githubusercontent.com/halinh1512/Predictive-Model-Semiconductor-SECOM/master/Data/secom_labels.data"

def load_secom(data_url, label_url):
    X = pd.read_csv(data_url, sep=r"\s+", header=None, na_values=["NaN"])
    labels = pd.read_csv(
        label_url,
        sep=r'\s+(?=")',
        engine="python",
        header=None,
        names=["label", "timestamp"]
    )
    labels["timestamp"] = labels["timestamp"].str.replace('"', '', regex=False)
    labels["timestamp"] = pd.to_datetime(labels["timestamp"], dayfirst=True)
    return X, labels

try:
    X, labels = load_secom(UCI_DATA, UCI_LABEL)
    source_used = "UCI Machine Learning Repository"
except Exception as e:
    print("UCI direct load failed. Fallback mirror를 사용합니다.")
    print("Reason:", e)
    X, labels = load_secom(MIRROR_DATA, MIRROR_LABEL)
    source_used = "GitHub mirror of UCI SECOM files"

print("Source:", source_used)
print("X shape:", X.shape)
print("Labels shape:", labels.shape)


### 여기서 볼 것

정상적으로 불러와졌다면 `X shape`은 대략 **(1567, 590)** 이 나와야 합니다.  
590개는 익명화된 공정/센서 측정 feature입니다.


In [ ]:
display(X.head())
display(labels.head())

print("\nLabel counts")
print(labels["label"].value_counts().sort_index())

print("\nLabel ratio (%)")
print((labels["label"].value_counts(normalize=True).sort_index() * 100).round(2))


## 2. Pass / Fail 불균형 확인

SECOM에서 `-1 = Pass`, `1 = Fail`입니다.

이 데이터는 Fail이 매우 적기 때문에 나중에 모델 평가에서 단순 Accuracy만 보면 안 됩니다.


In [ ]:
label_names = labels["label"].map({-1: "Pass", 1: "Fail"})
counts = label_names.value_counts().reindex(["Pass", "Fail"])

ax = counts.plot(kind="bar")
ax.set_title("SECOM Pass / Fail Distribution")
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print(counts)


## 3. 결측치 확인

실제 공정 데이터에서는 센서 미수집, 통신 문제, 특정 공정 단계 미측정 등 여러 이유로 결측치가 생길 수 있습니다.

여기서는 **왜 결측됐는지는 알 수 없으므로 원인을 단정하지 않고**, 데이터 품질 관점에서 결측률만 확인합니다.


In [ ]:
missing_count = X.isna().sum()
missing_rate = (missing_count / len(X) * 100).sort_values(ascending=False)

print("전체 결측값 수:", int(X.isna().sum().sum()))
print("결측치가 하나라도 있는 feature 수:", int((missing_count > 0).sum()))
print("결측률 50% 이상 feature 수:", int((missing_rate >= 50).sum()))

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_rate_pct": missing_count / len(X) * 100
}).sort_values("missing_rate_pct", ascending=False)

display(missing_summary.head(15))


In [ ]:
top_missing = missing_summary.head(15).sort_values("missing_rate_pct")

ax = top_missing["missing_rate_pct"].plot(kind="barh")
ax.set_title("Top 15 Features by Missing Rate")
ax.set_xlabel("Missing rate (%)")
ax.set_ylabel("Feature index")
plt.tight_layout()
plt.show()


## 4. 상수 변수 확인

모든 샘플에서 값이 같은 변수는 Pass/Fail을 구분하는 정보가 없습니다.  
다음 단계의 전처리에서 제거 후보가 됩니다.


In [ ]:
nunique = X.nunique(dropna=True)
constant_cols = nunique[nunique <= 1].index.tolist()

print("상수 feature 수:", len(constant_cols))
print("상수 feature index 예시:", constant_cols[:20])


## 5. Step 1 요약

아래 셀은 포트폴리오에 사용할 수 있도록 데이터 상태를 한 번에 요약합니다.


In [ ]:
summary = pd.DataFrame({
    "item": [
        "samples",
        "features",
        "pass_count",
        "fail_count",
        "missing_values_total",
        "features_with_missing",
        "features_missing_50pct_or_more",
        "constant_features"
    ],
    "value": [
        X.shape[0],
        X.shape[1],
        int((labels["label"] == -1).sum()),
        int((labels["label"] == 1).sum()),
        int(X.isna().sum().sum()),
        int((X.isna().sum() > 0).sum()),
        int(((X.isna().mean() * 100) >= 50).sum()),
        len(constant_cols)
    ]
})

display(summary)

print("\nStep 1 완료.")
print("다음 단계: 결측률 기준과 상수 변수 제거 원칙을 정한 뒤, 데이터 누수 없이 train/test를 분리합니다.")
